In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path

# 设置样式（在字体设置之前）
import seaborn as sns
sns.set_style("whitegrid")
sns.set_palette("husl")

# 尝试加载本地中文字体（相对路径）
font_path = Path('fonts/SourceHanSansSC-Regular.otf')

if font_path.exists():
    # 使用本地字体
    fm.fontManager.addfont(str(font_path))
    font_prop = fm.FontProperties(fname=str(font_path))
    font_name = font_prop.get_name()
    plt.rcParams['font.sans-serif'] = [font_name]
    plt.rcParams['axes.unicode_minus'] = False
    print(f"字体设置完成: {font_name}")
else:
    print("提示: 中文字体未配置，图表中文可能显示异常")
    print(f"      请下载思源黑体到 {font_path}")
    print("      下载地址: https://github.com/adobe-fonts/source-han-sans/releases")

In [ ]:
# 黄金定投策略收益范围分析
# 本工具用于分析黄金AU9999定投策略的收益范围，帮助理解不同定投周期和起始时间对收益的影响

# 安装必要的库（如果尚未安装）
# #!pip install akshare pandas numpy matplotlib seaborn -q

import akshare as ak
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Tuple, List

print("库导入成功！")

## 1. 获取AU9999历史数据

In [ ]:
def fetch_au9999_data() -> pd.DataFrame:
    """获取上海黄金交易所AU9999历史数据"""
    print("正在获取AU9999历史数据...")
    df = ak.spot_hist_sge(symbol='Au99.99')
    
    # 转换日期格式
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    print(f"数据获取成功！共 {len(df)} 条记录")
    print(f"日期范围: {df['date'].min().date()} 至 {df['date'].max().date()}")
    print(f"最新收盘价: {df['close'].iloc[-1]:.2f} 元/克")
    
    return df

# 获取数据
df_gold = fetch_au9999_data()

# 显示最近5天数据
print("\n最近5天数据:")
df_gold.tail()

## 2. 定投策略模拟器

In [ ]:
class GoldDIPSimulator:
    """黄金定投策略模拟器（按周定投）"""
    
    def __init__(self, price_data: pd.DataFrame):
        """
        初始化模拟器
        
        参数:
            price_data: 价格数据，需包含 date 和 close 列
        """
        self.price_data = price_data.copy()
        self.price_data = self.price_data[['date', 'open', 'close', 'high', 'low']]
    
    def simulate_dip(
        self,
        start_date: str,
        end_date: str,
        weekly_investment: float = 100,
        invest_weekday: int = 0
    ) -> dict:
        """
        模拟按周定投策略
        
        参数:
            start_date: 开始日期 (YYYY-MM-DD)
            end_date: 结束日期 (YYYY-MM-DD)
            weekly_investment: 每周定投金额（元）
            invest_weekday: 每周星期几定投 (0=周一, 1=周二, ..., 6=周日)
        
        返回:
            包含定投结果统计的字典
        """
        start = pd.to_datetime(start_date)
        end = pd.to_datetime(end_date)
        
        # 筛选日期范围
        mask = (self.price_data['date'] >= start) & (self.price_data['date'] <= end)
        period_data = self.price_data[mask].copy()
        
        if len(period_data) == 0:
            return {"error": "所选日期范围内无数据"}
        
        # 生成定投日期序列：找到第一个符合的星期几
        first_day = start
        while first_day.weekday() != invest_weekday:
            first_day += pd.Timedelta(days=1)
        
        invest_dates = pd.date_range(start=first_day, end=end, freq='W-MON') + pd.Timedelta(days=invest_weekday)
        
        # 执行定投
        total_invested = 0
        total_grams = 0
        invest_records = []
        
        for invest_date in invest_dates:
            if invest_date > end:
                break
            
            # 找到定投日的价格（T+1确认，但用定投日净值计算）
            price_data = period_data[period_data['date'] >= invest_date]
            if len(price_data) == 0:
                continue
            
            actual_date = price_data['date'].iloc[0]
            price = price_data['close'].iloc[0]
            
            # 计算购买克数
            grams = weekly_investment / price
            
            total_invested += weekly_investment
            total_grams += grams
            
            invest_records.append({
                'date': actual_date,
                'price': price,
                'amount': weekly_investment,
                'grams': grams
            })
        
        if total_grams == 0:
            return {"error": "未能执行任何定投操作"}
        
        # 计算期末价值
        final_price = period_data['close'].iloc[-1]
        final_value = total_grams * final_price
        
        # 计算收益
        profit = final_value - total_invested
        return_rate = (profit / total_invested) * 100
        
        return {
            'start_date': start_date,
            'end_date': end_date,
            'invest_count': len(invest_records),
            'total_invested': total_invested,
            'total_grams': total_grams,
            'avg_cost': total_invested / total_grams,
            'final_price': final_price,
            'final_value': final_value,
            'profit': profit,
            'return_rate': return_rate,
            'records': invest_records
        }
    
    def rolling_window_analysis(
        self,
        holding_weeks: list = None,
        weekly_investment: float = 100,
        invest_weekday: int = 0
    ) -> pd.DataFrame:
        """
        滚动窗口分析：计算所有可能的定投起始组合的收益率
        
        参数:
            holding_weeks: 持有周数列表，如 [12, 24, 48, 96, 192]
            weekly_investment: 每周定投金额
            invest_weekday: 每周星期几定投 (0=周一, 1=周二, ..., 6=周日)
        
        返回:
            包含所有模拟结果的DataFrame
        """
        results = []
        
        for weeks in holding_weeks:
            print(f"正在分析持有 {weeks} 周的定投策略...")
            
            # 计算滚动窗口
            for i in range(len(self.price_data) - weeks):
                start_row = self.price_data.iloc[i]
                end_idx = i + weeks
                
                if end_idx >= len(self.price_data):
                    continue
                
                end_row = self.price_data.iloc[end_idx]
                
                start_date = start_row['date']
                end_date = end_row['date']
                
                # 计算定投收益
                result = self.simulate_dip(
                    start_date=start_date.strftime('%Y-%m-%d'),
                    end_date=end_date.strftime('%Y-%m-%d'),
                    weekly_investment=weekly_investment,
                    invest_weekday=invest_weekday
                )
                
                if 'error' not in result:
                    results.append({
                        'holding_weeks': weeks,
                        'start_date': start_date,
                        'end_date': end_date,
                        'total_invested': result['total_invested'],
                        'final_value': result['final_value'],
                        'profit': result['profit'],
                        'return_rate': result['return_rate'],
                        'avg_cost': result['avg_cost'],
                        'final_price': result['final_price']
                    })
        
        return pd.DataFrame(results)

print("定投模拟器类已定义！（按周定投）")

## 3. 可视化函数

In [ ]:
def plot_return_distribution(results_df: pd.DataFrame, holding_weeks: int):
    """绘制特定持有期收益率分布"""
    data = results_df[results_df['holding_weeks'] == holding_weeks]
    
    if len(data) == 0:
        print(f"没有持有期为 {holding_weeks} 周的数据")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'黄金定投 {holding_weeks} 周收益率分布分析', fontsize=16, fontweight='bold')
    
    # 1. 收益率直方图
    ax1 = axes[0, 0]
    ax1.hist(data['return_rate'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(data['return_rate'].mean(), color='red', linestyle='--', linewidth=2, label=f"平均: {data['return_rate'].mean():.2f}%")
    ax1.axvline(0, color='green', linestyle='-', linewidth=1, alpha=0.5)
    ax1.set_xlabel('收益率 (%)')
    ax1.set_ylabel('频数')
    ax1.set_title('收益率分布')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. 盈亏比例
    ax2 = axes[0, 1]
    profit_count = (data['return_rate'] > 0).sum()
    loss_count = (data['return_rate'] <= 0).sum()
    colors = ['#2ecc71' if profit_count > loss_count else '#e74c3c']
    ax2.pie([profit_count, loss_count], labels=[f'盈利\n{profit_count}次', f'亏损\n{loss_count}次'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
    ax2.set_title(f'盈利概率: {profit_count/len(data)*100:.1f}%')
    
    # 3. 收益率时间序列
    ax3 = axes[1, 0]
    ax3.plot(data['start_date'], data['return_rate'], marker='o', markersize=2, linewidth=1)
    ax3.axhline(0, color='green', linestyle='-', linewidth=1, alpha=0.5)
    ax3.fill_between(data['start_date'], data['return_rate'], 0, 
                     where=(data['return_rate'] >= 0), color='#2ecc71', alpha=0.3, label='盈利')
    ax3.fill_between(data['start_date'], data['return_rate'], 0, 
                     where=(data['return_rate'] < 0), color='#e74c3c', alpha=0.3, label='亏损')
    ax3.set_xlabel('开始定投日期')
    ax3.set_ylabel('收益率 (%)')
    ax3.set_title('收益率随时间变化')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45)
    
    # 4. 统计指标
    ax4 = axes[1, 1]
    ax4.axis('off')
    stats_text = f"""
【统计指标】
━━━━━━━━━━━━━━━━━━━━━━
样本数量:      {len(data)}
平均收益率:   {data['return_rate'].mean():+.2f}%
中位数收益率: {data['return_rate'].median():+.2f}%
最大收益:     {data['return_rate'].max():+.2f}%
最大亏损:     {data['return_rate'].min():+.2f}%
标准差:       {data['return_rate'].std():.2f}%
盈利次数:     {profit_count} ({profit_count/len(data)*100:.1f}%)
亏损次数:     {loss_count} ({loss_count/len(data)*100:.1f}%)
    """
    ax4.text(0.1, 0.5, stats_text, fontsize=12,
             verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

def plot_holding_period_comparison(results_df: pd.DataFrame):
    """比较不同持有期的收益率"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('不同持有期收益率对比', fontsize=16, fontweight='bold')
    
    # 按持有期分组统计
    summary = results_df.groupby('holding_weeks')['return_rate'].agg([
        ('mean', 'mean'),
        ('median', 'median'),
        ('min', 'min'),
        ('max', 'max'),
        ('std', 'std'),
        ('count', 'count'),
        ('profit_ratio', lambda x: (x > 0).sum() / len(x) * 100)
    ]).reset_index()
    
    weeks = summary['holding_weeks'].values
    x = np.arange(len(weeks))
    width = 0.35
    
    # 1. 平均收益率和中位数
    ax1 = axes[0, 0]
    ax1.bar(x - width/2, summary['mean'], width, label='平均', color='steelblue', alpha=0.8)
    ax1.bar(x + width/2, summary['median'], width, label='中位数', color='coral', alpha=0.8)
    ax1.axhline(0, color='black', linestyle='-', linewidth=0.8)
    ax1.set_xlabel('持有周数')
    ax1.set_ylabel('收益率 (%)')
    ax1.set_title('平均收益率 vs 中位数')
    ax1.set_xticks(x)
    ax1.set_xticklabels([f'{w}周' for w in weeks])
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # 2. 收益率范围
    ax2 = axes[0, 1]
    ax2.fill_between(x, summary['min'], summary['max'], alpha=0.3, color='gray', label='收益范围')
    ax2.plot(x, summary['mean'], marker='o', linewidth=2, label='平均收益', color='red')
    ax2.axhline(0, color='black', linestyle='-', linewidth=0.8)
    ax2.set_xlabel('持有周数')
    ax2.set_ylabel('收益率 (%)')
    ax2.set_title('收益率范围')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xticks(x)
    ax2.set_xticklabels([f'{w}周' for w in weeks])
    
    # 3. 盈利概率
    ax3 = axes[1, 0]
    colors = ['#2ecc71' if x >= 50 else '#e67e22' if x >= 40 else '#e74c3c' for x in summary['profit_ratio']]
    ax3.bar(x, summary['profit_ratio'], color=colors, alpha=0.8)
    ax3.axhline(50, color='black', linestyle='--', linewidth=1, alpha=0.5, label='50%')
    ax3.set_xlabel('持有周数')
    ax3.set_ylabel('盈利概率 (%)')
    ax3.set_title('定投盈利概率')
    ax3.set_xticks(x)
    ax3.set_xticklabels([f'{w}周' for w in weeks])
    ax3.set_ylim([0, 100])
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. 统计表格
    ax4 = axes[1, 1]
    ax4.axis('off')
    table_data = []
    for _, row in summary.iterrows():
        table_data.append([
            f"{row['holding_weeks']}周",
            f"{row['mean']:+.1f}%",
            f"{row['profit_ratio']:.1f}%",
            f"{row['min']:+.1f}% ~ {row['max']:+.1f}%"
        ])
    
    table = ax4.table(cellText=table_data, 
                     colLabels=['持有期', '平均收益', '盈利概率', '收益范围'],
                     cellLoc='center',
                     loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    # 表头
    for i in range(4):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    # 数据行
    for i in range(1, len(table_data) + 1):
        for j in range(4):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#E7E6E6')
    
    ax4.set_title('统计汇总', pad=20)
    
    plt.tight_layout()
    plt.show()
    
    return summary

print("可视化函数已定义！")

In [ ]:
# 获取今日日期（动态）
from datetime import date
today = date.today()
today_str = today.strftime('%Y-%m-%d')

# 设置分析日期范围：2025-06-01 至今
start_date = '2025-06-01'
end_date = today_str  # 动态使用今日日期
weekly_investment = 100  # 每周定投金额（元）
invest_weekday = 0  # 每周星期几定投 (0=周一, 1=周二, ..., 6=周日)

# 基于日期范围自动计算持有期（周数）
start = pd.to_datetime(start_date)
end = pd.to_datetime(end_date)
total_weeks = (end - start).days // 7
# 生成从12周到总周数的持有期列表（间隔1周，细颗粒度分析）
holding_weeks = list(range(12, total_weeks + 1, 1))

print(f"分析日期范围: {start_date} 至 {end_date}")
print(f"今日日期: {today_str}")
print(f"总周数: {total_weeks} 周")
print(f"持有期分析: 从 12 周到 {total_weeks} 周，共 {len(holding_weeks)} 个持有期")

In [ ]:
# 过滤数据：只保留 2022-02-24 之后的数据
cutoff_date = pd.to_datetime('2022-02-24')
df_gold_filtered = df_gold[df_gold['date'] >= cutoff_date].reset_index(drop=True)

# 创建模拟器
simulator = GoldDIPSimulator(df_gold_filtered)

print("模拟器已创建！")
print(f"原始数据范围: {df_gold['date'].min().date()} 至 {df_gold['date'].max().date()}")
print(f"过滤后数据范围: {df_gold_filtered['date'].min().date()} 至 {df_gold_filtered['date'].max().date()}")
print(f"过滤后数据条数: {len(df_gold_filtered)} 条")

### 4.1 单次定投模拟示例

In [ ]:
# 使用设置参数运行单次模拟
result = simulator.simulate_dip(
    start_date=start_date,
    end_date=end_date,
    weekly_investment=weekly_investment,
    invest_weekday=invest_weekday
)

print("定投结果:")
print(f"  开始日期: {result['start_date']}")
print(f"  结束日期: {result['end_date']}")
print(f"  定投次数: {result['invest_count']} 次")
print(f"  总投入: {result['total_invested']:.2f} 元")
print(f"  累计黄金: {result['total_grams']:.2f} 克")
print(f"  平均成本: {result['avg_cost']:.2f} 元/克")
print(f"  期末价格: {result['final_price']:.2f} 元/克")
print(f"  期末价值: {result['final_value']:.2f} 元")
print(f"  总收益: {result['profit']:+.2f} 元")
print(f"  收益率: {result['return_rate']:+.2f}%")

### 4.2 滚动窗口分析（可调整参数）

In [ ]:
# 运行滚动窗口分析（使用自动计算的持有期）
results_df = simulator.rolling_window_analysis(
    holding_weeks=holding_weeks,
    weekly_investment=weekly_investment,
    invest_weekday=invest_weekday
)

print(f"\n分析完成！共生成 {len(results_df)} 个样本")
results_df.head()

### 4.3 可视化结果

In [ ]:
# 显示中间持有期的详细分析（自动选择）
selected_weeks = holding_weeks[len(holding_weeks) // 2]  # 选择中间的持有期

if selected_weeks in results_df['holding_weeks'].values:
    plot_return_distribution(results_df, selected_weeks)
else:
    print(f"警告：{selected_weeks}周不在分析结果中")

In [ ]:
# 不同持有期对比
summary = plot_holding_period_comparison(results_df)

## 5. 自定义测试

**说明**：下方单元格提供了一个可定制的定投模拟。你可以修改以下参数来测试不同场景：

| 参数 | 说明 | 示例值 |
|------|------|--------|
| `start_date` | 定投开始日期 | `'2025-06-01'` |
| `end_date` | 定投结束日期 | `today_str`（动态获取今日日期） |
| `weekly_investment` | 每周定投金额（元） | `200` |
| `invest_weekday` | 每周星期几定投（0=周一...6=周日） | `0` |

**操作方法**：在下方代码单元格中修改参数后运行即可查看结果。

In [ ]:
# ===== 自定义测试区域：修改下方参数进行测试 =====

# 设置测试参数（可修改）
test_start_date = '2025-06-01'      # 定投开始日期
test_end_date = today_str            # 定投结束日期（使用动态今日日期）
test_weekly_investment = 200         # 每周定投金额（元）
test_invest_weekday = 0              # 定投星期几 (0=周一, ..., 6=周日)

# 运行模拟
custom_result = simulator.simulate_dip(
    start_date=test_start_date,
    end_date=test_end_date,
    weekly_investment=test_weekly_investment,
    invest_weekday=test_invest_weekday
)

# 显示结果
print("自定义测试定投结果:")
print(f"  开始日期: {custom_result['start_date']}")
print(f"  结束日期: {custom_result['end_date']}")
print(f"  定投次数: {custom_result['invest_count']} 次")
print(f"  总投入: {custom_result['total_invested']:.2f} 元")
print(f"  累计黄金: {custom_result['total_grams']:.2f} 克")
print(f"  平均成本: {custom_result['avg_cost']:.2f} 元/克")
print(f"  期末价格: {custom_result['final_price']:.2f} 元/克")
print(f"  期末价值: {custom_result['final_value']:.2f} 元")
print(f"  总收益: {custom_result['profit']:+.2f} 元")
print(f"  收益率: {custom_result['return_rate']:+.2f}%")

## 6. 均线智能定投策略分析

基于500日移动平均线（约2年）的智能定投策略：
- 价格高于均线时减少投入（逢高减仓）
- 价格低于均线时增加投入（逢低加仓）

In [ ]:
class SmartGoldDIPSimulator(GoldDIPSimulator):
    """黄金智能定投策略模拟器（基于均线的智能定投）"""
    
    def __init__(self, price_data: pd.DataFrame, ma_period: int = 500):
        """
        初始化智能定投模拟器
        
        参数:
            price_data: 价格数据，需包含 date 和 close 列
            ma_period: 均线周期（交易日），默认500日约等于2年
        """
        super().__init__(price_data)
        self.ma_period = ma_period
        # 计算移动平均线
        self.price_data['ma'] = self.price_data['close'].rolling(window=ma_period).mean()
    
    def _calculate_investment_multiplier(self, current_price: float, ma_price: float) -> Tuple[float, str]:
        """
        根据价格与均线的偏离程度计算投资倍数
        
        参数:
            current_price: 当前价格
            ma_price: 均线价格
        
        返回:
            (投资倍数, 分档描述)
        """
        if pd.isna(ma_price):
            return 1.0, "均线未形成"
        
        deviation = (current_price - ma_price) / ma_price * 100
        
        if deviation > 0:
            # 高于均线：减少投资
            if deviation <= 15:
                return 0.9, "高于均线 0-15%"
            elif deviation <= 50:
                return 0.8, "高于均线 15-50%"
            elif deviation <= 100:
                return 0.7, "高于均线 50-100%"
            else:
                return 0.6, "高于均线 100%以上"
        else:
            # 低于均线：增加投资
            deviation = abs(deviation)
            if deviation <= 15:
                return 1.1, "低于均线 0-15%"
            elif deviation <= 30:
                return 1.3, "低于均线 15-30%"
            elif deviation <= 40:
                return 1.6, "低于均线 30-40%"
            else:
                return 2.1, "低于均线 40%以上"
    
    def simulate_smart_dip(
        self,
        start_date: str,
        end_date: str,
        base_weekly_investment: float = 100,
        invest_weekday: int = 0
    ) -> dict:
        """
        模拟基于均线的智能定投策略
        
        参数:
            start_date: 开始日期 (YYYY-MM-DD)
            end_date: 结束日期 (YYYY-MM-DD)
            base_weekly_investment: 基准每周定投金额（元），实际投入会根据价格偏离调整
            invest_weekday: 每周星期几定投 (0=周一, 1=周二, ..., 6=周日)
        
        返回:
            包含定投结果统计的字典
        """
        start = pd.to_datetime(start_date)
        end = pd.to_datetime(end_date)
        
        # 筛选日期范围
        mask = (self.price_data['date'] >= start) & (self.price_data['date'] <= end)
        period_data = self.price_data[mask].copy()
        
        if len(period_data) == 0:
            return {"error": "所选日期范围内无数据"}
        
        # 生成定投日期序列：找到第一个符合的星期几
        first_day = start
        while first_day.weekday() != invest_weekday:
            first_day += pd.Timedelta(days=1)
        
        invest_dates = pd.date_range(start=first_day, end=end, freq='W-MON') + pd.Timedelta(days=invest_weekday)
        
        # 执行智能定投
        total_invested = 0
        total_grams = 0
        invest_records = []
        tier_stats = {}  # 各分档投入统计
        
        for invest_date in invest_dates:
            if invest_date > end:
                break
            
            # 找到定投日的价格
            price_data = period_data[period_data['date'] >= invest_date]
            if len(price_data) == 0:
                continue
            
            actual_date = price_data['date'].iloc[0]
            price = price_data['close'].iloc[0]
            ma_price = price_data['ma'].iloc[0]
            
            # 计算投资倍数
            multiplier, tier_desc = self._calculate_investment_multiplier(price, ma_price)
            
            # 实际投入金额
            actual_investment = base_weekly_investment * multiplier
            
            # 计算购买克数
            grams = actual_investment / price
            
            total_invested += actual_investment
            total_grams += grams
            
            # 统计各分档投入
            if tier_desc not in tier_stats:
                tier_stats[tier_desc] = {'count': 0, 'amount': 0}
            tier_stats[tier_desc]['count'] += 1
            tier_stats[tier_desc]['amount'] += actual_investment
            
            invest_records.append({
                'date': actual_date,
                'price': price,
                'ma': ma_price,
                'multiplier': multiplier,
                'tier': tier_desc,
                'base_amount': base_weekly_investment,
                'actual_amount': actual_investment,
                'grams': grams
            })
        
        if total_grams == 0:
            return {"error": "未能执行任何定投操作"}
        
        # 计算期末价值
        final_price = period_data['close'].iloc[-1]
        final_value = total_grams * final_price
        
        # 计算收益
        profit = final_value - total_invested
        return_rate = (profit / total_invested) * 100
        
        # 计算如果用普通定投的情况
        normal_total = base_weekly_investment * len(invest_records)
        
        return {
            'start_date': start_date,
            'end_date': end_date,
            'ma_period': self.ma_period,
            'invest_count': len(invest_records),
            'total_invested': total_invested,
            'normal_total_invested': normal_total,
            'total_grams': total_grams,
            'avg_cost': total_invested / total_grams,
            'final_price': final_price,
            'final_value': final_value,
            'profit': profit,
            'return_rate': return_rate,
            'tier_stats': tier_stats,
            'records': invest_records
        }
    
    def compare_strategies(
        self,
        start_date: str,
        end_date: str,
        weekly_investment: float = 100,
        invest_weekday: int = 0
    ) -> dict:
        """
        对比普通定投和智能定投策略
        
        返回:
            包含两种策略结果的对比字典
        """
        # 普通定投
        normal_result = self.simulate_dip(
            start_date=start_date,
            end_date=end_date,
            weekly_investment=weekly_investment,
            invest_weekday=invest_weekday
        )
        
        # 智能定投
        smart_result = self.simulate_smart_dip(
            start_date=start_date,
            end_date=end_date,
            base_weekly_investment=weekly_investment,
            invest_weekday=invest_weekday
        )
        
        if 'error' in normal_result or 'error' in smart_result:
            return {"error": "策略对比失败"}
        
        # 计算差异
        profit_diff = smart_result['profit'] - normal_result['profit']
        return_diff = smart_result['return_rate'] - normal_result['return_rate']
        investment_diff = smart_result['total_invested'] - normal_result['total_invested']
        
        return {
            'normal': normal_result,
            'smart': smart_result,
            'profit_diff': profit_diff,
            'return_diff': return_diff,
            'investment_diff': investment_diff,
            'smart_advantage': profit_diff > 0
        }

# 创建智能定投模拟器（500日均线，约2年）
smart_simulator = SmartGoldDIPSimulator(df_gold, ma_period=500)
print(f"智能定投模拟器已创建！均线周期: {smart_simulator.ma_period}日")
print(f"智能定投分档规则:")
print("  | 高于均线的涨幅 | 实际扣款率 | 投资倍数 |")
print("  | :------ | :---- | :--- |")
print("  | 0-15%   | 90%   | 0.9x |")
print("  | 15-50%  | 80%   | 0.8x |")
print("  | 50-100% | 70%   | 0.7x |")
print("  | 100% 以上 | 60%   | 0.6x |")
print()
print("  | 低于均线的跌幅 | 实际扣款率 | 投资倍数 |")
print("  | :------ | :---- | :--- |")
print("  | 0-15%   | 110%  | 1.1x |")
print("  | 15-30%  | 130%  | 1.3x |")
print("  | 30-40%  | 160%  | 1.6x |")
print("  | 40% 以上  | 210%  | 2.1x |")

In [ ]:
def plot_strategy_comparison(comparison: dict, price_data: pd.DataFrame):
    """绘制普通定投和智能定投策略对比图"""
    normal = comparison['normal']
    smart = comparison['smart']
    
    # 获取分析期间的价格数据
    start = pd.to_datetime(normal['start_date'])
    end = pd.to_datetime(normal['end_date'])
    mask = (price_data['date'] >= start) & (price_data['date'] <= end)
    period_data = price_data[mask].copy()
    
    fig = plt.figure(figsize=(16, 12))
    
    # 1. 价格走势与定投点
    ax1 = plt.subplot(3, 2, (1, 2))
    ax1.plot(period_data['date'], period_data['close'], label='收盘价', linewidth=1.5, color='black')
    
    # 绘制移动平均线
    if 'ma' in period_data.columns:
        ax1.plot(period_data['date'], period_data['ma'], label=f'{smart["ma_period"]}日均线', linewidth=1.5, color='orange', linestyle='--')
    
    # 标记智能定投点
    smart_records = smart['records']
    smart_dates = [r['date'] for r in smart_records]
    smart_prices = [r['price'] for r in smart_records]
    multipliers = [r['multiplier'] for r in smart_records]
    
    # 用颜色表示倍数
    scatter = ax1.scatter(smart_dates, smart_prices, c=multipliers, cmap='RdYlGn', 
                         s=80, alpha=0.7, edgecolors='black', linewidth=0.5, label='定投点')
    plt.colorbar(scatter, ax=ax1, label='投资倍数')
    
    ax1.set_xlabel('日期')
    ax1.set_ylabel('价格 (元/克)')
    ax1.set_title(f'价格走势与定投点 ({normal["start_date"]} 至 {normal["end_date"]})', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)
    
    # 2. 策略对比表格
    ax2 = plt.subplot(3, 2, 3)
    ax2.axis('off')
    
    table_data = [
        ['指标', '普通定投', '智能定投', '差异'],
        ['定投次数', f"{normal['invest_count']}", f"{smart['invest_count']}", 
         f"{'0' if normal['invest_count'] == smart['invest_count'] else '—'}"],
        ['总投入(元)', f"{normal['total_invested']:.2f}", f"{smart['total_invested']:.2f}", 
         f"{comparison['investment_diff']:+.2f}"],
        ['累计黄金(克)', f"{normal['total_grams']:.2f}", f"{smart['total_grams']:.2f}", 
         f"{smart['total_grams'] - normal['total_grams']:+.2f}"],
        ['平均成本', f"{normal['avg_cost']:.2f}", f"{smart['avg_cost']:.2f}", 
         f"{smart['avg_cost'] - normal['avg_cost']:+.2f}"],
        ['期末价值(元)', f"{normal['final_value']:.2f}", f"{smart['final_value']:.2f}", 
         f"{smart['final_value'] - normal['final_value']:+.2f}"],
        ['总收益(元)', f"{normal['profit']:+.2f}", f"{smart['profit']:+.2f}", 
         f"{comparison['profit_diff']:+.2f}"],
        ['收益率', f"{normal['return_rate']:+.2f}%", f"{smart['return_rate']:+.2f}%", 
         f"{comparison['return_diff']:+.2f}%"]
    ]
    
    table = ax2.table(cellText=table_data, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.8)
    
    # 表头样式
    for i in range(4):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # 数据行样式
    for i in range(1, len(table_data)):
        for j in range(4):
            if j == 3:  # 差异列
                val = table_data[i][j]
                if '+' in val:
                    table[(i, j)].set_facecolor('#C6EFCE')
                    table[(i, j)].set_text_props(color='#006100', weight='bold')
                elif '-' in val:
                    table[(i, j)].set_facecolor('#FFC7CE')
                    table[(i, j)].set_text_props(color='#9C0006', weight='bold')
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#E7E6E6')
    
    ax2.set_title('策略对比汇总', fontsize=12, fontweight='bold')
    
    # 3. 收益率对比柱状图
    ax3 = plt.subplot(3, 2, 4)
    strategies = ['普通定投', '智能定投']
    returns = [normal['return_rate'], smart['return_rate']]
    colors = ['#5B9BD5' if r >= 0 else '#ED7D31' for r in returns]
    
    bars = ax3.bar(strategies, returns, color=colors, alpha=0.8, edgecolor='black', linewidth=1)
    ax3.axhline(0, color='black', linestyle='-', linewidth=0.8)
    ax3.set_ylabel('收益率 (%)')
    ax3.set_title('收益率对比', fontsize=12, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 在柱子上标注数值
    for bar, ret in zip(bars, returns):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{ret:+.2f}%', ha='center', va='bottom' if height >= 0 else 'top', fontweight='bold')
    
    # 4. 智能定投分档投入统计
    ax4 = plt.subplot(3, 2, 5)
    
    tier_stats = smart['tier_stats']
    tier_names = list(tier_stats.keys())
    tier_amounts = [tier_stats[t]['amount'] for t in tier_names]
    tier_counts = [tier_stats[t]['count'] for t in tier_names]
    
    # 按投入金额排序
    sorted_indices = sorted(range(len(tier_amounts)), key=lambda i: tier_amounts[i], reverse=True)
    tier_names = [tier_names[i] for i in sorted_indices]
    tier_amounts = [tier_amounts[i] for i in sorted_indices]
    
    colors = ['#FF6B6B' if '高于' in t else '#4ECDC4' for t in tier_names]
    bars = ax4.barh(tier_names, tier_amounts, color=colors, alpha=0.8, edgecolor='black', linewidth=1)
    
    ax4.set_xlabel('投入金额 (元)')
    ax4.set_title('智能定投各分档投入分布', fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='x')
    
    # 标注数值
    for bar, amount, count in zip(bars, tier_amounts, tier_counts):
        width = bar.get_width()
        ax4.text(width, bar.get_y() + bar.get_height()/2,
                f'{amount:.0f}元 ({count}次)', ha='left', va='center', fontsize=9)
    
    # 5. 累计投入和持仓对比
    ax5 = plt.subplot(3, 2, 6)
    ax5.axis('off')
    
    # 计算每期累计情况
    normal_records = normal['records']
    smart_records = smart['records']
    
    normal_cumulative = []
    smart_cumulative = []
    
    cumul_normal = 0
    cumul_smart = 0
    
    # 按日期对齐两种策略的记录
    max_len = max(len(normal_records), len(smart_records))
    
    for i in range(max_len):
        if i < len(normal_records):
            cumul_normal += normal_records[i]['grams']
        if i < len(smart_records):
            cumul_smart += smart_records[i]['grams']
        
        normal_cumulative.append(cumul_normal)
        smart_cumulative.append(cumul_smart)
    
    x = range(len(normal_cumulative))
    
    line1, = ax5.plot(x, normal_cumulative, label='普通定投', linewidth=2, color='#5B9BD5', marker='o', markersize=3)
    line2, = ax5.plot(x, smart_cumulative, label='智能定投', linewidth=2, color='#ED7D31', marker='s', markersize=3)
    
    ax5.set_xlabel('定投期数')
    ax5.set_ylabel('累计黄金 (克)')
    ax5.set_title('累计持仓对比', fontsize=12, fontweight='bold')
    ax5.legend(loc='upper left')
    ax5.grid(True, alpha=0.3)
    
    # 添加最终数值标注
    ax5.text(len(x)-1, normal_cumulative[-1], f' {normal_cumulative[-1]:.2f}克', 
             fontsize=10, va='center', color='#5B9BD5', fontweight='bold')
    ax5.text(len(x)-1, smart_cumulative[-1], f' {smart_cumulative[-1]:.2f}克', 
             fontsize=10, va='center', color='#ED7D31', fontweight='bold')
    
    plt.suptitle(f'定投策略对比分析 ({normal["start_date"]} 至 {normal["end_date"]})', 
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

print("plot_strategy_comparison 函数已定义！")

In [ ]:
# 获取今日日期（动态）
from datetime import date, timedelta

today = date.today()
today_str = today.strftime('%Y-%m-%d')

# 设置分析窗口：2025-06-01 至今
ma_analysis_start = '2025-06-01'
ma_analysis_end = today_str

print(f"均线智能定投分析窗口: {ma_analysis_start} 至 {ma_analysis_end}")
print(f"今日日期: {today_str}")

# 运行策略对比
comparison = smart_simulator.compare_strategies(
    start_date=ma_analysis_start,
    end_date=ma_analysis_end,
    weekly_investment=100,
    invest_weekday=0
)

print("\n" + "="*60)
print("策略对比结果")
print("="*60)

print("\n【普通定投策略】")
print(f"  定投次数: {comparison['normal']['invest_count']} 次")
print(f"  总投入: {comparison['normal']['total_invested']:.2f} 元")
print(f"  累计黄金: {comparison['normal']['total_grams']:.2f} 克")
print(f"  平均成本: {comparison['normal']['avg_cost']:.2f} 元/克")
print(f"  期末价值: {comparison['normal']['final_value']:.2f} 元")
print(f"  总收益: {comparison['normal']['profit']:+.2f} 元")
print(f"  收益率: {comparison['normal']['return_rate']:+.2f}%")

print("\n【均线智能定投策略】")
print(f"  均线周期: {comparison['smart']['ma_period']}日")
print(f"  定投次数: {comparison['smart']['invest_count']} 次")
print(f"  总投入: {comparison['smart']['total_invested']:.2f} 元")
print(f"  累计黄金: {comparison['smart']['total_grams']:.2f} 克")
print(f"  平均成本: {comparison['smart']['avg_cost']:.2f} 元/克")
print(f"  期末价值: {comparison['smart']['final_value']:.2f} 元")
print(f"  总收益: {comparison['smart']['profit']:+.2f} 元")
print(f"  收益率: {comparison['smart']['return_rate']:+.2f}%")

print("\n【差异分析】")
print(f"  收益差异: {comparison['profit_diff']:+.2f} 元")
print(f"  收益率差异: {comparison['return_diff']:+.2f}%")
print(f"  投入差异: {comparison['investment_diff']:+.2f} 元")
if comparison['smart_advantage']:
    print(f"  结论: 智能定投策略更优，多盈利 {comparison['profit_diff']:.2f} 元")
else:
    print(f"  结论: 普通定投策略更优，智能定投少盈利 {abs(comparison['profit_diff']):.2f} 元")

# 显示各分档投入统计
print("\n【智能定投分档投入统计】")
for tier, stats in comparison['smart']['tier_stats'].items():
    avg_amount = stats['amount'] / stats['count']
    print(f"  {tier}: {stats['count']}次, 总投入{stats['amount']:.2f}元, 平均{avg_amount:.2f}元/次")

# 绘制对比图（使用包含ma列的数据）
plot_strategy_comparison(comparison, smart_simulator.price_data)